<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day4/Session3/session3_finetune_lora.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 3 — LoRA: Fine-Tuning That Actually Fits

**Day 4 · 13:45 – 15:15**

Freeze 99.8% of the model, train the other 0.2%, and run it for real on a free GPU.

**Runtime → Change runtime type → T4 GPU**

> This notebook is **self-contained** — it reloads everything, so it works even if
> Colab restarted over lunch.

## Cell 1 — Setup and load the model

In [ ]:
!pip install -q transformers datasets accelerate peft

import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2-0.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto")

print("Loaded. GPU memory: %.2f GB" % (torch.cuda.memory_allocated(0)/1024**3))

## Cell 2 — Record the baseline (again, if you restarted)

If you still have `before` from Session 2, skip this cell. If Colab restarted, run it —
you cannot compare later without it.

In [ ]:
def ask(m, question, max_new_tokens=120):
    messages = [
        {"role": "system", "content": "You are a helpful Ethiopian customer service assistant."},
        {"role": "user",   "content": question},
    ]
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs,
                         max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


QUESTIONS = [
    "ስለ ምርታችሁ ዋጋ ማወቅ እፈልጋለሁ።",
    "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?",
    "ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?",
]

model.eval()
before = {}
for q in QUESTIONS:
    before[q] = ask(model, q)
    print("Q:", q)
    print("A:", before[q][:180])
    print("-" * 60)

## Cell 3 — Load and format the data

In [ ]:
from datasets import load_dataset

raw = load_dataset("addisai/FineTome-single-turn-dedup-amharic", split="train")

def to_text(ex):
    turns = ex.get("conversations_amharic") or []
    if len(turns) < 2:
        return {"text": ""}
    q = (turns[0] or {}).get("content", "")
    a = (turns[1] or {}).get("content", "")
    if not q or not a:
        return {"text": ""}
    return {"text": f"### ጥያቄ:\n{q}\n\n### መልስ:\n{a}{tok.eos_token}"}

N = 1000
data = raw.select(range(min(N, len(raw)))).map(to_text, remove_columns=raw.column_names)
data = data.filter(lambda x: len(x["text"]) > 40)

print(f"{len(data)} training examples")
print(data[0]["text"][:250])

## Cell 4 — Tokenize

Note: we do **not** pad here. We let the collator pad each batch to its own longest
example — that is much faster than padding everything to 512.

In [ ]:
MAX_LEN = 384

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = data.map(tokenize, batched=True, remove_columns=["text"])
print(train_ds)
print("first example length:", len(train_ds[0]["input_ids"]), "tokens")

## Cell 5 — What is LoRA, in one cell?

**The problem:** full fine-tuning changes all 494 million numbers. Each one needs a
gradient plus two optimizer values → about 16 bytes per parameter → **~8 GB** just for
the bookkeeping.

**The trick:** freeze the whole model. Add small extra matrices ("adapters") beside the
attention layers. Train *only* those.

| | Full fine-tuning | LoRA |
|---|---|---|
| Numbers trained | 494,000,000 | ~1,000,000 |
| Memory | ~8 GB | ~2 GB |
| Result file | ~1 GB | ~5 MB |

The three settings you will touch:

- **`r` (rank) = 16** — adapter capacity. Higher learns more, costs more. 8–32 is normal.
- **`lora_alpha` = 32** — how strongly the adapter affects the model. Common rule: 2 × r.
- **`target_modules`** — which layers get adapters. `q_proj` and `v_proj` are the standard choice.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

**Read that output.** You should see roughly:

```
trainable params: 1,081,344 || all params: 495,114,240 || trainable%: 0.2184
```

**0.2%.** That is the whole idea. The other 99.8% is frozen and costs no gradient memory.

## Cell 6 — Training settings

Small numbers on purpose, so this finishes inside the lesson.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# For causal language models: mlm=False. The collator also handles padding
# each batch dynamically and builds the labels for us.
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

args = TrainingArguments(
    output_dir="./qwen-amharic-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,      # effective batch size = 8
    learning_rate=2e-4,
    warmup_steps=20,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collator,
)

print("Ready to train on", len(train_ds), "examples")
print("Expect roughly 5-15 minutes on a T4.")

## Cell 7 — Train

**While this runs, watch the `loss` column.**

| What you see | What it means |
|---|---|
| Loss **falling** (e.g. 2.5 → 1.6) | Learning. This is what you want. |
| Loss **flat** | Learning rate too low, or the data is broken |
| Loss **jumping wildly** | Learning rate too high |
| Loss = `nan` | Numerical blow-up — restart and lower the learning rate |

Do **not** close the browser tab — Colab disconnects and you lose the run.

In [ ]:
model.train()

t0 = time.time()
result = trainer.train()
minutes = (time.time() - t0) / 60

print()
print("Training finished in %.1f minutes" % minutes)
print("Final training loss: %.4f" % result.training_loss)
print("GPU memory used: %.2f GB" % (torch.cuda.max_memory_allocated(0)/1024**3))

### Note the peak memory

It should be well under 15 GB. Full fine-tuning this same model would have needed
about 8 GB *just for the optimizer* — plus activations. LoRA is why this fit.

## Cell 8 — Save the adapter and see how small it is

In [ ]:
OUT = "./qwen-amharic-lora"

model.save_pretrained(OUT)
tok.save_pretrained(OUT)

print("Saved to", OUT)
print()
!du -sh ./qwen-amharic-lora
!ls -la ./qwen-amharic-lora

**A few megabytes.** The base model is over 900 MB, but everything you *taught* it
lives in that tiny file.

This is how real teams work: keep one copy of the base model, then one small adapter
per language, per task, per customer.

> ### Download it before you leave
> Colab deletes everything when the session ends.
> In the file panel on the left: right-click the `qwen-amharic-lora` folder → **Download**.

## Cell 9 — Quick look: did anything change?

The full comparison is Session 4, but here is a first taste.

In [ ]:
model.eval()

q = QUESTIONS[1]
print("QUESTION:", q)
print()
print("BEFORE:", before[q][:200])
print()
print("AFTER :", ask(model, q)[:200])

## Cell 10 — Save the baseline for Session 4

In [ ]:
import pickle

with open("baseline.pkl", "wb") as f:
    pickle.dump({"questions": QUESTIONS, "before": before}, f)

print("Saved baseline.pkl")
print("Download it from the file panel if you might restart before Session 4.")

---

## What you learned

| Idea | Why it matters |
|---|---|
| **Freeze most, train little** | 0.2% of the parameters, most of the benefit |
| **Loss falling = learning** | The one number to watch during training |
| **An adapter is ~5 MB** | One base model, many tiny adapters |
| **Small settings finish in class** | Real projects use all 83,000 examples and run for hours — same code |

**Next:** Session 4 — the honest before/after comparison, and publishing your adapter.